In [0]:
from pyspark.sql.functions import *


In [0]:
fact_table = "retailnova.gold.fact_sales"

In [0]:

orders = spark.table("retailnova.silver.orders")
customers = spark.table("retailnova.gold.dim_customer")
products = spark.table("retailnova.gold.dim_product")
stores = spark.table("retailnova.gold.dim_store")
dates = spark.table("retailnova.gold.dim_date")

In [0]:
fact_df = orders.join(
    customers,
    orders.customer_id == customers.customer_id,
    "left"
).join(
    products,
    orders.product_id == products.product_id,
    "left"
).join(
    stores,
    orders.store_id == stores.store_id,
    "left"
).join(
    dates,
    to_date(orders.order_timestamp) == dates.full_date,
    "left"
).select(
    orders.order_id,
    customers.customer_key,
    products.product_key,
    stores.store_key,
    dates.date_key,
    orders.order_timestamp,
    orders.quantity,
    orders.unit_price,
    orders.discount_amount,
    orders.status,
    orders.payment_method
)

In [0]:
fact_df = fact_df.withColumn(
    "sales_amount",
    (col("quantity") * col("unit_price")) - col("discount_amount")
)

display(fact_df)

print("Fact sales records:", fact_df.count())

order_id,customer_key,product_key,store_key,date_key,order_timestamp,quantity,unit_price,discount_amount,status,payment_method,sales_amount
O0551695,50415,20598,79,20260522,2026-05-22T07:46:03.000Z,4.0,693.58,688.32,Completed,Net Banking,2086.0
O0551702,36497,20469,162,20260310,2026-03-10T09:04:48.000Z,4.0,818.31,81.74,Completed,Credit Card,3191.5
O0551706,40343,10968,103,20260219,2026-02-19T16:08:36.000Z,3.0,279.78,76.14,Completed,UPI,763.1999999999999
O0551807,32984,15044,137,20260806,2026-08-06T17:27:15.000Z,5.0,1643.88,931.15,Completed,UPI,7288.250000000002
O0551808,75695,16297,417,20260714,2026-07-14T08:56:22.000Z,5.0,1537.11,1051.96,Completed,Credit Card,6633.589999999999
O0551857,36905,12215,377,20260218,2026-02-18T22:26:31.000Z,2.0,321.08,81.69,Completed,Net Banking,560.47
O0551866,37940,16150,492,20260426,2026-04-26T15:21:56.000Z,2.0,1366.7,614.1,Shipped,Cash,2119.3
O0551939,88657,11451,448,20260418,2026-04-18T08:21:44.000Z,1.0,3800.99,335.34,Completed,UPI,3465.6499999999996
O0552013,69485,19467,448,20260723,2026-07-23T22:50:18.000Z,4.0,3637.63,2817.83,Completed,UPI,11732.69
O0552041,22268,17494,170,20260816,2026-08-16T06:35:51.000Z,3.0,1511.88,117.7,Completed,UPI,4417.9400000000005


Fact sales records: 1048879


In [0]:
# Create table on first run
if not spark.catalog.tableExists(fact_table):

    fact_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(fact_table)

    print("Fact Sales table created and loaded")

else:

    # Existing order IDs
    existing_orders = spark.table(fact_table).select(
        "order_id"
    ).distinct()

    # Only new orders
    new_fact_df = fact_df.join(
        existing_orders,
        fact_df.order_id == existing_orders.order_id,
        "left_anti"
    )

    new_count = new_fact_df.count()

    print("New fact records:", new_count)

    # Append only new orders
    if new_count > 0:

        new_fact_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(fact_table)

    print("Fact Sales updated successfully")


New fact records: 0
Fact Sales updated successfully


In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM retailnova.gold.fact_sales;

total_records
1048879


In [0]:
%sql
SELECT *
FROM retailnova.gold.fact_sales
LIMIT 10;

sales_key,order_id,customer_key,product_key,store_key,date_key,order_timestamp,quantity,unit_price,discount_amount,sales_amount,status,payment_method
1,O0551695,50415,20598,79,20260522,2026-05-22T07:46:03.000Z,4.0,693.58,688.32,2086.0,Completed,Net Banking
2,O0551702,36497,20469,162,20260310,2026-03-10T09:04:48.000Z,4.0,818.31,81.74,3191.5,Completed,Credit Card
3,O0551706,40343,10968,103,20260219,2026-02-19T16:08:36.000Z,3.0,279.78,76.14,763.1999999999999,Completed,UPI
4,O0551807,32984,15044,137,20260806,2026-08-06T17:27:15.000Z,5.0,1643.88,931.15,7288.250000000002,Completed,UPI
5,O0551808,75695,16297,417,20260714,2026-07-14T08:56:22.000Z,5.0,1537.11,1051.96,6633.589999999999,Completed,Credit Card
6,O0551857,36905,12215,377,20260218,2026-02-18T22:26:31.000Z,2.0,321.08,81.69,560.47,Completed,Net Banking
7,O0551866,37940,16150,492,20260426,2026-04-26T15:21:56.000Z,2.0,1366.7,614.1,2119.3,Shipped,Cash
8,O0551939,88657,11451,448,20260418,2026-04-18T08:21:44.000Z,1.0,3800.99,335.34,3465.6499999999996,Completed,UPI
9,O0552013,69485,19467,448,20260723,2026-07-23T22:50:18.000Z,4.0,3637.63,2817.83,11732.69,Completed,UPI
10,O0552041,22268,17494,170,20260816,2026-08-16T06:35:51.000Z,3.0,1511.88,117.7,4417.9400000000005,Completed,UPI
